In [1]:
import requests

In [17]:
league="mens-college-basketball"
id='401809115'

In [ ]:
r = requests.get(f'https://cdn.espn.com/core/{league}/playbyplay?xhr=1&gameId={id}')

In [29]:
# Core API returns ALL plays (CDN only returns recent). Set limit high to get everything in one request.
url = f'https://sports.core.api.espn.com/v2/sports/basketball/leagues/{league}/events/{id}/competitions/{id}/plays?limit=300'
r = requests.get(url)
print(f'Status: {r.status_code}')
data = r.json()
print(f'Total plays: {data.get("count")} | Returned: {len(data.get("items", []))}')

Status: 200
Total plays: 200 | Returned: 200


In [30]:
import json     
json.loads(r.text)

{'$ref': 'http://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/events/401809115/competitions/401809115/plays?lang=en&region=us',
 'count': 200,
 'pageIndex': 1,
 'pageSize': 300,
 'pageCount': 1,
 'items': [{'$ref': 'http://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball/events/401809115/competitions/401809115/plays/401809115118670338?lang=en&region=us',
   'id': '401809115118670338',
   'sequenceNumber': '118670338',
   'type': {'id': '615', 'text': 'Jumpball'},
   'text': 'Start game',
   'shortText': 'Start game',
   'alternativeText': 'Start game',
   'shortAlternativeText': 'Start game',
   'awayScore': 0,
   'homeScore': 0,
   'period': {'number': 1, 'displayValue': '1st Half'},
   'clock': {'value': 1200.0, 'displayValue': '20:00'},
   'valid': True,
   'scoringPlay': False,
   'priority': False,
   'scoreValue': 0,
   'modified': '2026-02-08T03:07Z',
   'probability': {'$ref': 'http://sports.core.api.espn.com/v2/spor

In [41]:
import pandas as pd
import re


def _extract_id(ref_url):
    """Extract the trailing numeric ID from a core API $ref URL."""
    if not ref_url:
        return None
    m = re.search(r'/(\d+)\?', ref_url) or re.search(r'/(\d+)$', ref_url)
    return m.group(1) if m else None


def get_plays(league, game_id, limit=300):
    """Fetch all play-by-play data for a game and return a DataFrame.

    Args:
        league:  ESPN league slug, e.g. 'mens-college-basketball', 'nba', 'nfl'
        game_id: ESPN event/game ID
        limit:   Max plays per request (default 300, usually enough for one game)

    Returns:
        pd.DataFrame with one row per play
    """
    url = (
        f'https://sports.core.api.espn.com/v2/sports/basketball/leagues/{league}'
        f'/events/{game_id}/competitions/{game_id}/plays?limit={limit}'
    )
    r = requests.get(url)
    r.raise_for_status()
    data = r.json()
    plays = data.get('items', [])

    rows = []
    for p in plays:
        team_ref = p.get('team', {}).get('$ref', '')
        participants = p.get('participants', [])

        rows.append({
            'id': p.get('id'),
            'sequence': p.get('sequenceNumber'),
            'period': p.get('period', {}).get('displayValue'),
            'clock': p.get('clock', {}).get('displayValue'),
            'text': p.get('text', ''),
            'type': p.get('type', {}).get('text'),
            'team_id': _extract_id(team_ref),
            'home_score': p.get('homeScore'),
            'away_score': p.get('awayScore'),
            'scoring_play': p.get('scoringPlay', False),
            'shooting_play': p.get('shootingPlay', False),
            'score_value': p.get('scoreValue', 0),
            'points_attempted': p.get('pointsAttempted', 0),
            'wallclock': p.get('wallclock'),
            'athlete_id': _extract_id(participants[0].get('athlete', {}).get('$ref', '')) if participants else None,
            'assist_athlete_id': _extract_id(participants[1].get('athlete', {}).get('$ref', '')) if len(participants) > 1 else None,
        })

    df = pd.DataFrame(rows)
    print(f'{len(df)} plays fetched for game {game_id}')
    return df


# Usage
df = get_plays(league, id)
df

281 plays fetched for game 401809115


,id,sequence,period,clock,text,type,team_id,home_score,away_score,scoring_play,shooting_play,score_value,points_attempted,wallclock,athlete_id,assist_athlete_id
0,401809115118670338,118670338,1st Half,20:00,Start game,Jumpball,NaN,0,0,False,False,0,0,2026-02-08T03:07:06Z,NaN,NaN
1,401809115118670339,118670339,1st Half,19:59,Jump Ball won by UC Irvine,Jumpball,300,0,0,False,False,0,0,2026-02-08T03:07:07Z,5107567,NaN
2,401809115118670360,118670360,1st Half,19:59,Jump Ball lost by UC Santa Barbara,Jumpball,2540,0,0,False,False,0,0,2026-02-08T03:07:07Z,5242262,NaN
3,401809115118670369,118670369,1st Half,19:46,Jurian Dixon makes 18-foot jumper (Tama Isaac ...,JumpShot,300,0,2,True,True,2,2,2026-02-08T03:07:20Z,4711267,5313007
4,401809115118670378,118670378,1st Half,19:28,Hosana Kitenge bad pass\nturnover,Lost Ball Turnover,2540,0,2,False,False,0,0,2026-02-08T03:07:37Z,4593883,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276,401809115118674048,118674048,2nd Half,11:36,Colin Smith Offensive Rebound.,Offensive Rebound,2540,52,39,False,False,0,0,2026-02-08T04:24:14Z,4684812,NaN
277,401809115118674051,118674051,2nd Half,11:31,Colin Smith makes 24-foot three point jumper (...,JumpShot,2540,55,39,True,True,3,3,2026-02-08T04:24:20Z,4684812,5241843
278,401809115118674055,118674055,2nd Half,11:13,Andre Henry misses layup,LayUpShot,300,55,39,False,True,2,2,2026-02-08T04:24:38Z,4702629,NaN
279,401809115118674056,118674056,2nd Half,11:13,Marvin McGhee IV Block.,Block Shot,2540,55,39,False,False,0,0,2026-02-08T04:24:38Z,5241843,NaN


In [35]:
def plays_to_text(df):
    """Convert play-by-play DataFrame to a single newline-separated string."""
    lines = []
    for _, row in df.iterrows():
        lines.append(f"{row['clock']} | {row['type']} | {row['team_id']} | {row['text']}")
    return '\n'.join(lines)

play_text = plays_to_text(df)
print(play_text[:500])

20:00 | Jumpball | nan | Start game
19:59 | Jumpball | 300 | Jump Ball won by UC Irvine
19:59 | Jumpball | 2540 | Jump Ball lost by UC Santa Barbara
19:46 | JumpShot | 300 | Jurian Dixon makes 18-foot jumper (Tama Isaac assists)
19:28 | Lost Ball Turnover | 2540 | Hosana Kitenge bad pass
turnover
19:28 | Steal | 300 | Akiva McBirney-Griffin Steal.
19:04 | JumpShot | 300 | Tama Isaac misses 11-foot floating jump shot
19:01 | Defensive Rebound | 2540 | Hosana Kitenge Defensive Rebound.
18:47 | Lay


In [ ]:
import openai
from openai import OpenAI
import dotenv
import os

dotenv.load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")


client = OpenAI()

def find_insights(play_by_play_data):
    prompt = """
    You are a sports analyst. 
    You are given a play-by-play dataset of a basketball game. 
    Your task is to analyze the data and provide insights into the game.

    """
    
    result = client.responses.create(
        model="gpt-5.1-codex-max",
        instructions=prompt,
        input=play_by_play_data,
        reasoning={ "effort": "low" },
    )

    return result


In [62]:
def play_by_play_summary():
    import datetime
    from datetime import datetime
    df = get_plays(league, id)
    play_by_play_data = plays_to_text(df)
    insights = find_insights(play_by_play_data)
    with open('full_game_summary.md', 'a') as f:
        f.write("summary at " + datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "\n")
        f.write(insights.output_text)
    print(insights.output_text)

In [61]:
import time
while True:
    play_by_play_summary()
    time.sleep(120)

300 plays fetched for game 401809115
Here's what jumps out from the play‑by‑play:

- **Halftime score/flow:** UC Santa Barbara led 34–28 at the break. UC Irvine got most of its first‑half points in the paint from Kyle Evans and Harrison Carrington; there were no made threes before the half. UCSB balanced inside play from Aidan Mahaney and Hosana Kitenge with perimeter shooting: Zion Sensley, Michael Simcoe and Colin Smith all hit from deep, and Smith chipped in a late three to get them to 34.

- **Start of the 2nd half:** The Gauchos came out hot with a 9–2 run capped by a Kitenge layup and a Shaw three to stretch the lead to double digits by the 16‑minute mark. By the 10:10 mark of the second half UCSB was up 55–39, outscoring Irvine 21–11 in that stretch.

- **Key performers:**
  - UCSB: Colin Smith has multiple buckets in each half, including two made threes, and hit the lone first‑half free throw. Aidan Mahaney is steady with floaters and a 25‑footer early in the second half. Hosan

KeyboardInterrupt: 